# 04 - Modeling

**Project**: Customer Churn Prediction & Prevention Dashboard

**Goal**: Train and compare machine learning models using enhanced features.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load the enhanced dataset
df = pd.read_csv('df_enhanced.csv')

print("Shape:", df.shape)

In [ ]:
# Prepare features and target
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

# One-Hot Encoding for categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("Encoding completed. Final shape:", X.shape)

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# Logistic Regression (Baseline)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_pred_proba_lr = lr.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))
print("ROC AUC:", round(roc_auc_score(y_test, y_pred_proba_lr), 4))

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))
print("ROC AUC:", round(roc_auc_score(y_test, y_pred_proba_rf), 4))

In [ ]:
# XGBoost
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_pred_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb))
print("ROC AUC:", round(roc_auc_score(y_test, y_pred_proba_xgb), 4))

In [ ]:
# Model Comparison
models = {
    "Logistic Regression": roc_auc_score(y_test, y_pred_proba_lr),
    "Random Forest": roc_auc_score(y_test, y_pred_proba_rf),
    "XGBoost": roc_auc_score(y_test, y_pred_proba_xgb)
}

comparison = pd.DataFrame.from_dict(models, orient='index', columns=['ROC AUC'])
print(comparison.sort_values(by='ROC AUC', ascending=False))

In [ ]:
# Feature Importance
import matplotlib.pyplot as plt

importances = pd.Series(xgb.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 8))
importances.plot(kind='barh')
plt.title('Top 15 Most Important Features (XGBoost)')
plt.show()